# 商场顾客细分分析

本Notebook展示了顾客细分分析的完整流程，包括数据探索、K-Means聚类和RFM分析。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

%matplotlib inline
sns.set_style('whitegrid')

## 1. 数据加载与清洗

支持两种数据格式：
- 商场顾客数据（包含 Annual Income, Spending Score）
- 销售交易数据（包含 InvoiceNo, CustomerID, Quantity, UnitPrice, InvoiceDate）

In [2]:
# 加载数据 - 请修改文件路径
file_path = '../data/Mall_Customers.csv'  # 或 'OnlineRetail.csv'

# 尝试多种编码
encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1']
df = None

for encoding in encodings:
    try:
        df = pd.read_csv(file_path, encoding=encoding)
        print(f'成功使用 {encoding} 编码读取文件')
        break
    except UnicodeDecodeError:
        continue

if df is None:
    raise ValueError('无法识别文件编码')

df.head()

## 2. 数据预处理

根据数据类型进行不同的预处理

In [3]:
# 检查数据类型
columns = df.columns.str.lower()

has_transaction_data = all(col in columns for col in ['invoiceno', 'customerid', 'quantity', 'unitprice', 'invoicedate'])
has_customer_data = all(col in columns for col in ['customerid', 'annual income', 'spending score'])

print(f'交易数据格式: {has_transaction_data}')
print(f'顾客数据格式: {has_customer_data}')

## 3. RFM 分析

对于交易数据，先计算 RFM 指标

In [4]:
def calculate_rfm(df):
    """计算 RFM 指标"""
    df = df.copy()
    
    # 数据清洗
    if 'customerid' in df.columns:
        df = df.dropna(subset=['customerid'])
        df = df[df['customerid'] != 0]
    
    if 'quantity' in df.columns and 'unitprice' in df.columns:
        df = df[df['quantity'] > 0]
        df = df[df['unitprice'] > 0]
        df['amount'] = df['quantity'] * df['unitprice']
    
    # 转换时间
    if 'invoicedate' in df.columns:
        df['invoicedate'] = pd.to_datetime(df['invoicedate'], errors='coerce')
        df = df.dropna(subset=['invoicedate'])
        
        snapshot_date = df['invoicedate'].max() + pd.Timedelta(days=1)
        
        rfm = df.groupby('customerid').agg({
            'invoicedate': lambda x: (snapshot_date - x.max()).days,
            'invoiceno': 'nunique',
            'amount': 'sum'
        })
        rfm.columns = ['recency', 'frequency', 'monetary']
        
        return rfm
    else:
        raise ValueError('缺少计算 RFM 所需的列')

# 如果是交易数据，计算 RFM
if has_transaction_data:
    rfm_df = calculate_rfm(df)
    print('RFM 数据计算完成')
    rfm_df.head()

## 4. 肘部法则选择最优 K

In [5]:
# 准备特征数据
if has_transaction_data and 'rfm_df' in dir():
    features = rfm_df[['recency', 'frequency', 'monetary']].values
else:
    features = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

# 标准化
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# 肘部法则
inertia = []
silhouette_scores = []
k_range = range(2, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(features_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(features_scaled, labels))

# 找到最优 K
optimal_k = silhouette_scores.index(max(silhouette_scores)) + 2
print(f'最优聚类数量: {optimal_k}')

In [6]:
# 绘制肘部法则图
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, 'bo-')
plt.xlabel('K 值')
plt.ylabel('Inertia')
plt.title('肘部法则图')

plt.subplot(1, 2, 2)
plt.bar(k_range, silhouette_scores, color='skyblue')
plt.xlabel('K 值')
plt.ylabel('轮廓系数')
plt.title('轮廓系数图')

plt.tight_layout()
plt.show()

## 5. K-Means 聚类分析

In [7]:
# 执行聚类
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
labels = kmeans.fit_predict(features_scaled)

# 添加聚类标签
if has_transaction_data and 'rfm_df' in dir():
    rfm_df['Cluster'] = labels
    result_df = rfm_df
else:
    df['Cluster'] = labels
    result_df = df

print('聚类完成')
result_df.head()

## 6. 聚类结果可视化

In [8]:
# PCA 降维可视化
pca = PCA(n_components=2)
pca_result = pca.fit_transform(features_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1], hue=labels, palette='Set1', s=100)
plt.title('PCA 聚类结果可视化')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(title='Cluster')
plt.show()

In [9]:
# 如果是顾客数据，绘制收入与消费评分的散点图
if has_customer_data:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', hue='Cluster', 
                    data=df, palette='Set1', s=100)
    plt.title('顾客聚类结果')
    plt.show()

## 7. 聚类统计分析

In [10]:
# 聚类统计
if has_transaction_data and 'rfm_df' in dir():
    stats = rfm_df.groupby('Cluster').agg({
        'recency': ['mean', 'min', 'max'],
        'frequency': ['mean', 'min', 'max'],
        'monetary': ['mean', 'min', 'max', 'count']
    }).round(2)
else:
    stats = df.groupby('Cluster').agg({
        'Annual Income (k$)': ['mean', 'min', 'max'],
        'Spending Score (1-100)': ['mean', 'min', 'max']
    }).round(2)

stats

## 8. 聚类描述

根据聚类结果描述不同顾客群体的特征

In [11]:
# 生成聚类描述
def describe_clusters(stats):
    descriptions = {}
    
    for cluster in stats.index:
        if has_transaction_data and 'rfm_df' in dir():
            r = stats.loc[cluster, 'recency']['mean']
            f = stats.loc[cluster, 'frequency']['mean']
            m = stats.loc[cluster, 'monetary']['mean']
            count = stats.loc[cluster, 'monetary']['count']
            
            if r < 30 and f > 10 and m > 1000:
                desc = f'高价值活跃客户 - 近期购买(R={r:.0f}天)、频繁购买(F={f:.1f}次)、高消费(M={m:.0f})'
            elif r < 30 and f < 5:
                desc = f'新客户 - 近期购买(R={r:.0f}天)、购买次数较少(F={f:.1f}次)'
            elif r > 180:
                desc = f'流失风险客户 - 长时间未购买(R={r:.0f}天)'
            elif m > 500:
                desc = f'高消费客户 - 平均消费(M={m:.0f})'
            else:
                desc = f'普通客户 - 平均R={r:.0f}天, F={f:.1f}次, M={m:.0f}'
        else:
            income = stats.loc[cluster, 'Annual Income (k$)']['mean']
            score = stats.loc[cluster, 'Spending Score (1-100)']['mean']
            
            if income > 80 and score > 60:
                desc = f'高收入高消费群体 - 收入(¥{income:.0f}k), 消费评分({score:.0f})'
            elif income > 80 and score < 40:
                desc = f'高收入低消费群体 - 收入(¥{income:.0f}k), 消费评分({score:.0f})'
            elif income < 40 and score > 60:
                desc = f'低收入高消费群体 - 收入(¥{income:.0f}k), 消费评分({score:.0f})'
            elif income < 40 and score < 40:
                desc = f'低收入低消费群体 - 收入(¥{income:.0f}k), 消费评分({score:.0f})'
            else:
                desc = f'中等收入群体 - 收入(¥{income:.0f}k), 消费评分({score:.0f})'
        
        descriptions[cluster] = desc
    
    return descriptions

descriptions = describe_clusters(stats)
for cluster, desc in descriptions.items():
    print(f'Cluster {cluster}: {desc}')

## 9. 保存结果

In [12]:
# 保存聚类结果
output_file = '../data/clustering_results.csv'
result_df.to_csv(output_file, encoding='utf-8-sig')
print(f'结果已保存到 {output_file}')

# 保存统计信息
stats_file = '../data/cluster_stats.csv'
stats.to_csv(stats_file, encoding='utf-8-sig')
print(f'统计信息已保存到 {stats_file}')